# Phân loại ảnh bằng SIFT + Bag of Visual Words + SVM

Notebook này được thiết kế để tải mã nguồn từ GitHub và chạy trực tiếp trên Kaggle.

Trước khi chọn **Run All**:

1. Bật Internet trong phần **Notebook options** để Git và pip có thể hoạt động.
2. Chọn **Add Input** để gắn bộ dữ liệu Caltech-101 đã có cấu trúc thư mục ảnh vào notebook.
3. Nếu repository là private, cần đổi `REPO_URL` sang URL có thông tin xác thực phù hợp. Không ghi token trực tiếp vào notebook công khai.

## 1. Cấu hình

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ngvihoa/bovw-image-classification.git"
BRANCH = "main"
PROJECT_DIR = Path("/kaggle/working/bovw-image-classification")
OUTPUT_DIR = Path("/kaggle/working/bovw_output")

# Giữ None để tự tìm thư mục chứa các lớp trong /kaggle/input.
# Hoặc nhập đường dẫn cụ thể, ví dụ:
# DATA_DIR = Path("/kaggle/input/caltech101/101_ObjectCategories")
DATA_DIR = None

CLASSES = ["airplanes", "Motorbikes", "Faces", "watch", "car_side"]
VOCAB_SIZE = 500
TEST_SAMPLE_COUNT = 10  # Số ảnh test được hiển thị sau đánh giá.

# Ảnh bên ngoài tập train/test. Upload ảnh bằng Add Input rồi nhập đường dẫn.
# Ví dụ: Path("/kaggle/input/my-external-image/example.jpg")
EXTERNAL_IMAGE_PATH = None
EXTERNAL_TRUE_LABEL = None  # Tùy chọn: ví dụ "airplanes".
SEED = 42

print("Project directory:", PROJECT_DIR)
print("Output directory :", OUTPUT_DIR)

## 2. Clone hoặc cập nhật repository

Ở lần chạy đầu tiên, cell này dùng `git clone`. Nếu thư mục dự án đã tồn tại do chạy lại notebook, cell sẽ dùng `git pull --ff-only`.

In [ ]:
import subprocess

def run_command(command, cwd=None):
    print("$", " ".join(map(str, command)))
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)

if (PROJECT_DIR / ".git").is_dir():
    run_command(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run_command(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    run_command(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)
else:
    run_command(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, PROJECT_DIR])

run_command(["git", "log", "-1", "--oneline"], cwd=PROJECT_DIR)

## 3. Cài đặt thư viện

Kaggle thường đã có NumPy, scikit-learn và OpenCV. Cell này vẫn kiểm tra file `requirements.txt` của repository và cài các gói còn thiếu hoặc chưa đúng phiên bản.

In [ ]:
import sys

requirements_file = PROJECT_DIR / "requirements.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Không tìm thấy {requirements_file}")

run_command([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_file])

## 4. Huấn luyện và đánh giá với K = 500

Cell này tự tìm thư mục dữ liệu trong `/kaggle/input` nếu `DATA_DIR` là `None`, sau đó sử dụng trực tiếp các mô-đun trong `src/` để chạy SIFT, BoVW và SVM với một kích thước từ điển duy nhất là K = 500.

In [ ]:
import os
import sys
import numpy as np

def find_data_directory(search_root):
    required = {name.casefold(): name for name in CLASSES}
    for current, directories, _files in os.walk(Path(search_root)):
        actual = {name.casefold(): name for name in directories}
        if set(required).issubset(actual):
            names = [actual[name.casefold()] for name in CLASSES]
            return Path(current), names
    return None, None

search_root = DATA_DIR if DATA_DIR is not None else Path("/kaggle/input")
DATA_DIR, detected_classes = find_data_directory(search_root)
if DATA_DIR is None:
    raise FileNotFoundError(
        f"Không tìm thấy thư mục chứa đủ các lớp {CLASSES} bên dưới {search_root}."
    )
CLASSES = detected_classes
print("Dữ liệu sử dụng:", DATA_DIR)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import config
from src.dataset import load_dataset
from src.sift import collect_training_descriptors
from src.vocabulary import build_vocabulary, save_vocabulary
from src.bovw import build_features, save_features
from src.classifier import train_classifier, save_model
from src.evaluate import compute_metrics, save_metrics

# Ghi kết quả vào /kaggle/working thay vì bên trong repository.
config.DATA_DIR = str(DATA_DIR)
config.CLASSES = list(CLASSES)
config.RANDOM_STATE = SEED
config.ARTIFACT_DIR = str(OUTPUT_DIR / "artifacts")
config.VOCAB_DIR = str(OUTPUT_DIR / "artifacts" / "vocabularies")
config.FEATURES_DIR = str(OUTPUT_DIR / "artifacts" / "features")
config.CLASSIFIERS_DIR = str(OUTPUT_DIR / "artifacts" / "classifiers")
config.RESULTS_DIR = str(OUTPUT_DIR / "results")
config.METRICS_DIR = str(OUTPUT_DIR / "results" / "metrics")

train_paths, test_paths, train_labels, test_labels = load_dataset(
    data_dir=str(DATA_DIR),
    classes=CLASSES,
    test_size=config.TEST_SIZE,
    random_state=SEED,
)
print(f"Train: {len(train_paths)} ảnh | Test: {len(test_paths)} ảnh")

print(f"\n{'=' * 70}\nBắt đầu huấn luyện với K={VOCAB_SIZE}")
np.random.seed(SEED)

descriptors = collect_training_descriptors(
    train_paths,
    max_descriptor_per_image=config.MAX_DESCRIPTOR_PER_IMAGE,
    max_total_descriptors=config.MAX_TOTAL_DESCRIPTORS,
)
vocabulary = build_vocabulary(descriptors, vocab_size=VOCAB_SIZE)
save_vocabulary(vocabulary, vocab_size=VOCAB_SIZE)

train_features = build_features(train_paths, vocabulary, vocab_size=VOCAB_SIZE)
test_features = build_features(test_paths, vocabulary, vocab_size=VOCAB_SIZE)
save_features(train_features, "train", vocab_size=VOCAB_SIZE)
save_features(test_features, "test", vocab_size=VOCAB_SIZE)

classifier = train_classifier(train_features, train_labels)
save_model(classifier, vocab_size=VOCAB_SIZE)
predictions = classifier.predict(test_features)
metrics = compute_metrics(test_labels, predictions)
save_metrics(metrics, vocab_size=VOCAB_SIZE)

print(f"K={VOCAB_SIZE}: {metrics}")
print("Đã hoàn tất. Kết quả tại:", OUTPUT_DIR)

## 5. Kết quả đánh giá

In [ ]:
import json

metrics_path = OUTPUT_DIR / "results" / "metrics" / f"metrics_{VOCAB_SIZE}.json"
with metrics_path.open(encoding="utf-8") as file:
    saved_metrics = json.load(file)

print(f"K              : {VOCAB_SIZE}")
print(f"Accuracy       : {saved_metrics['accuracy']:.4f}")
print(f"Macro precision: {saved_metrics['precision']:.4f}")
print(f"Macro recall   : {saved_metrics['recall']:.4f}")
print(f"Macro F1-score : {saved_metrics['f1_score']:.4f}")

## 6. Hiển thị kết quả trên 10 ảnh test

Cell dưới đây ưu tiên chọn cân bằng giữa dự đoán đúng và sai (mặc định 5 đúng, 5 sai), giúp quan sát lỗi của mô hình rõ hơn. Nếu không đủ ảnh sai, các vị trí còn lại được bù bằng ảnh đúng. Việc chọn ảnh sử dụng `SEED` nên có thể tái lập.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

sample_count = min(TEST_SAMPLE_COUNT, len(test_paths))
rng = np.random.default_rng(SEED)
labels_array = np.asarray(test_labels)
predictions_array = np.asarray(predictions)
correct_indices = np.flatnonzero(labels_array == predictions_array)
incorrect_indices = np.flatnonzero(labels_array != predictions_array)

incorrect_count = min(sample_count // 2, len(incorrect_indices))
correct_count = min(sample_count - incorrect_count, len(correct_indices))
selected_incorrect = rng.choice(
    incorrect_indices, size=incorrect_count, replace=False
)
selected_correct = rng.choice(
    correct_indices, size=correct_count, replace=False
)
sample_indices = np.concatenate([selected_incorrect, selected_correct])

# Bù thêm mẫu nếu một trong hai nhóm không đủ số lượng.
remaining_count = sample_count - len(sample_indices)
if remaining_count > 0:
    unused_indices = np.setdiff1d(
        np.arange(len(test_paths)), sample_indices, assume_unique=False
    )
    extra_indices = rng.choice(unused_indices, size=remaining_count, replace=False)
    sample_indices = np.concatenate([sample_indices, extra_indices])
rng.shuffle(sample_indices)

print(
    f"Tập test có {len(correct_indices)} dự đoán đúng và "
    f"{len(incorrect_indices)} dự đoán sai."
)
column_count = 5
row_count = int(np.ceil(sample_count / column_count))
figure, axes = plt.subplots(row_count, column_count, figsize=(18, 4 * row_count))
axes = np.asarray(axes).reshape(-1)

for axis, sample_index in zip(axes, sample_indices):
    image_path = Path(test_paths[int(sample_index)])
    true_label = CLASSES[int(test_labels[int(sample_index)])]
    predicted_label = CLASSES[int(predictions[int(sample_index)])]
    is_correct = true_label == predicted_label

    with Image.open(image_path) as image:
        axis.imshow(image.convert("RGB"))
    axis.axis("off")
    axis.set_title(
        f"Thật: {true_label}\nDự đoán: {predicted_label} "
        f"({'ĐÚNG' if is_correct else 'SAI'})",
        color="green" if is_correct else "red",
        fontsize=11,
    )

for axis in axes[sample_count:]:
    axis.axis("off")

figure.suptitle(f"Kết quả dự đoán trên {sample_count} ảnh test", fontsize=16)
figure.tight_layout()
plt.show()

## 7. Dự đoán một ảnh bên ngoài tập train/test

Đặt `EXTERNAL_IMAGE_PATH` trong cell cấu hình thành đường dẫn của một ảnh khác được thêm qua **Add Input**, sau đó chạy cell này. `EXTERNAL_TRUE_LABEL` là tùy chọn nếu đã biết nhãn thật. Cell sẽ từ chối ảnh thuộc tập train hoặc test hiện tại.

In [ ]:
from src.sift import extract_sift
from src.bovw import image_to_bovw

EXTERNAL_IMAGE_PATH = Path(
    "/kaggle/input/my-external-image/example.jpg"
)
EXTERNAL_TRUE_LABEL = None

def predict_external_image(image_path, true_label=None):
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy ảnh: {image_path}")

    known_paths = {
        Path(path).resolve() for path in [*train_paths, *test_paths]
    }
    if image_path.resolve() in known_paths:
        raise ValueError(
            "Ảnh này thuộc tập train hoặc test hiện tại; hãy chọn một ảnh bên ngoài."
        )
    if true_label is not None and true_label not in CLASSES:
        raise ValueError(f"Nhãn thật phải thuộc một trong các lớp: {CLASSES}")

    descriptors = extract_sift(str(image_path))
    feature = image_to_bovw(
        descriptors, vocabulary, vocab_size=VOCAB_SIZE
    ).reshape(1, -1)
    predicted_id = int(classifier.predict(feature)[0])
    predicted_label = CLASSES[predicted_id]

    with Image.open(image_path) as image:
        display_image = image.convert("RGB")
        plt.figure(figsize=(8, 6))
        plt.imshow(display_image)
        plt.axis("off")

    if true_label is None:
        title = f"Dự đoán: {predicted_label}"
        color = "black"
    else:
        is_correct = predicted_label == true_label
        title = (
            f"Nhãn thật: {true_label} | Dự đoán: {predicted_label} | "
            f"{'ĐÚNG' if is_correct else 'SAI'}"
        )
        color = "green" if is_correct else "red"

    plt.title(title, color=color, fontsize=14)
    plt.show()
    print("Ảnh bên ngoài:", image_path)
    print("Nhãn dự đoán:", predicted_label)
    return predicted_label

if EXTERNAL_IMAGE_PATH is None:
    print("Hãy đặt EXTERNAL_IMAGE_PATH trong cell cấu hình rồi chạy lại cell này.")
else:
    external_prediction = predict_external_image(
        EXTERNAL_IMAGE_PATH, true_label=EXTERNAL_TRUE_LABEL
    )